# **`Lluís Mompó Ruiz`**

# **Data cleaning and processing**



The dataset `simpsons_episodes.csv` was cleaned using **OpenRefine** before performing the visual analysis.  
The objective of the cleaning process was to remove irrelevant information, handle missing values, and ensure that all variables were stored in the correct format for visualization.

---

## 0. Creation of the project

The previous step before data cleaning is importing the dataset. In order to do it, we select columns are separated by commas, use character " to enclose cells containing column separators and trim leading & trailing whitespace from strings

## 1. Duplicated Rows

Duplicated rows were checked using the `id` column as the **primary key**, since each episode should have a unique identifier.

### Steps

All → **Edit rows → Remove duplicates (based on column `id`)**

### Result

No duplicated rows were found.  

---

## 2. Handling Missing Values

Missing values were identified using OpenRefine facets to each column.

### Steps

Select a column button → Facet → **Customized facets → Facet by null**

Then, go to the Facet/Filter section and check which columns have null values

### Results

- **3 null values in `imdb_rating`**
- **6 null values in `us_viewers_in_millions`**
- **1 null value in `videos_url`, `views` and `image_url` for the same observation.**

---

### 2.1 Removing Rows with Missing Ratings

The three rows with missing `imdb_rating` values also had missing viewer data.  
Since **ratings and viewers are the main variables used in the visualizations**, these rows were removed to avoid introducing artificial data into the analysis.

Additionally, an inspection of the dataset revealed that these three episodes belong to **Season 28**. In the dataset, this season contains only **four episodes**, meaning that most of its information is missing. Keeping the remaining single episode would make the season poorly represented and potentially introduce bias in the analysis.

For this reason, the three rows with missing ratings were removed, along with the remaining episode from the same season. As a result, **Season 28 was excluded from the dataset**.

### Steps

Facet/Filter (left side) → `imdb_rating` → **True**

All → **Edit rows → Remove matching rows**

The remaining episode from Season 28 was then filtered and removed to maintain consistency in the dataset.

id column → Sort → Numbers (to find the episode, it is the last)

Star → All → Facet by Star → Facet/Filter (left side) → Starred rows` → **True** → All → Remove matching rows

---

### 2.2 Imputing Missing Viewer Values

Three additional rows contained missing values only in `us_viewers_in_millions`.

These episodes belonged to the same season and were not special episodes (such as season finales or *Treehouse of Horror* episodes). Therefore, the missing values were imputed using the **average number of viewers for that season**, assuming that episodes within the same season tend to have similar audience levels.

### Steps

Facet/Filter → `viewers_millions` → **True**

Edit cell → Enter value **14.6** → **Apply to all identical cells**

---

### 2.3 Missing values for one observation

As we do not need these columns for the assignment, we leave them as null values. If they were needed, the different URLs can be found on internet.

---

## 3. Data Type Conversion

By default, OpenRefine imports all columns as **strings**.  
Each column was converted to the appropriate data type based on the information it contains.

### Numeric columns

- `id`
- `imdb_rating`
- `imdb_votes`
- `number_in_season`
- `season`
- `us_viewers_in_millions`
- `views`
- `number_in_series`
- `original_air_year`

### Date column

- `original_air_date`

Converting `original_air_date` to a **date format** allows aggregation by year, month, or weekday when creating visualizations.

---

## 4. Text Consistency Checks

To ensure the quality of textual data, several checks were performed on the `Title` column.

### 4.1 Trim Whitespaces

Leading and trailing whitespaces were removed from the `Title` column.

### 4.2 Clustering

Clustering was applied to detect titles that might represent the same episode but with minor spelling variations.

### Results

- No extra whitespaces were found.
- No title clusters indicating spelling inconsistencies were detected.

---

## 5. Column Removal

Several columns were removed because they were not relevant for the analysis or duplicated information already present in other fields.

Removed columns:

- `image_url`
- `production_code`
- `video_url`
- `views`
- `number_in_series`
- `title`
- `original_air_year`

### Reasons

- `image_url` and `video_url` contain multimedia links that are not useful for statistical analysis or visualization.
- `production_code` is not relevant to the research questions.
- `views` are not required for the charts to be produced.
- `number_in_series` duplicates the information already contained in the `id` column.
- `title` is not needed for visual analysis. If the user wants to know the title, the table can be used. 
- `original_air_year` is not used as we use the season column for temporal purposes.


### Steps in OpenRefine

Select column → **Edit column → Remove column**

---

## 6. Column Renaming

Once the columns that are needed for the project are selected, the columns were renamed to improve clarity and readability.

| Original Name | New Name |
|---------------|----------|
| id | ID |
| season | Season |
| number_in_season | Episode_Number |
| original_air_date | Date |
| imdb_rating | Rating |
| imdb_votes | Votes |
| us_viewers_in_millions | Viewers |

---

## 7. Extracting the Weekday

To analyze whether the number of viewers is related to the weekday on which the episode aired, a new column containing the **weekday** was created from the date.

### Steps

Date → **Edit column → Add column based on this column**

Formula:

`value.toDate().toString("EEEE")`


This creates a new column containing the day of the week (e.g., Monday, Tuesday, etc.).

# **Design and implementations**

### Imports

In [1]:
import altair as alt
import pandas as pd

### Loading data

After cleaning and preprocessing, data is loaded. It is sorted by ID and it is checked that it is loaded correctly

In [2]:
df = pd.read_csv("../data/simpsons-episodes-clean.csv", parse_dates=["Date"])
df = df.sort_values(by="ID")
print(df.head())
print(df.info())
print(df.describe())

     ID  Rating  Viewers  Votes  Number_season                      Date  \
238   1     8.2     26.7   3734              1 1989-12-17 00:00:00+00:00   
239   2     7.8     24.5   1973              2 1990-01-14 00:00:00+00:00   
547   3     7.5     27.5   1709              3 1990-01-21 00:00:00+00:00   
240   4     7.8     20.2   1701              4 1990-01-28 00:00:00+00:00   
522   5     8.1     27.1   1732              5 1990-02-04 00:00:00+00:00   

    Weekday  Season  
238  Sunday       1  
239  Sunday       1  
547  Sunday       1  
240  Sunday       1  
522  Sunday       1  
<class 'pandas.core.frame.DataFrame'>
Index: 596 entries, 238 to 572
Data columns (total 8 columns):
 #   Column         Non-Null Count  Dtype              
---  ------         --------------  -----              
 0   ID             596 non-null    int64              
 1   Rating         596 non-null    float64            
 2   Viewers        596 non-null    float64            
 3   Votes          596 non-nu

### Previous considerations:

To analyze how ratings evolved over time, seasons were used as the temporal unit instead of calendar years. Television series are naturally organized by seasons, and episodes from the same season may span two different years. Therefore, using seasons provides a more meaningful comparison.

### How have the ratings evolved over time?

Charts:

- Line chart and heatmap (Juxtaposition)

- Vertical alignment. Same x-axis (Season)

**Line chart:**

- Provides an overview of the global trend in episode ratings.

- Axes: Season (x-axis, ordinal) and Average rating (y-axis, quantitative).

- Average values are labeled and marked with red points for improving readability and help identify the value associated with each season.

- The Y axis is truncated since the focus is on relative trends.

**Heatmap:**

- Represents individual episode ratings within each season. Helping to identify highly rated episodes or periods where ratings decreased

- Axes: Season (x-axis, ordinal) and Episode number (y-axis, ordinal)

- Color is used to encode rating values. The Viridis color scheme is used to ensure accessibility for colorblind viewers.

- Labels are shown only for episodes with notably high or low ratings (>8.5 / <6.5), allowing the viewer to quickly identify outliers without overloading the chart. White colors for darker ones and black colors for lighter ones

Considered options:

- Scatter line chart for individual ratings. Problem: it was cluttered because of the high number of episodes

In [3]:
season_avg_rating = df.groupby("Season")["Rating"].mean().reset_index()

season_rating = alt.Chart(season_avg_rating).mark_line().encode(
    x=alt.X("Season:O", title="Season", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("Rating:Q", title="Average Rating", scale = alt.Scale(domain=[6,9])),
)

text_rating = alt.Chart(season_avg_rating).mark_text(dy=-10,fontSize=10).encode(
    x=alt.X("Season:O"),
    y=alt.Y("Rating:Q"),
    text=alt.Text('Rating:Q', format='.1f')
)

point_rating = alt.Chart(season_avg_rating).mark_circle(color='red').encode(
    x=alt.X("Season:O"),
    y=alt.Y("Rating:Q")
)

rating_chart = (season_rating + text_rating + point_rating).properties(
    title="Average Rating per Season",
    width=900,
    height=300
)

heatmap_rating = alt.Chart(df).mark_rect(stroke='white', strokeWidth=0.5).encode(
    y=alt.Y('Number_season:O', title='Episode Number'),
    x=alt.X('Season:O', title='Season', axis = alt.Axis(orient='top',labelAngle=0)),
    color=alt.Color('Rating:Q', title='Rating',scale=alt.Scale(scheme='viridis'),legend=alt.Legend(orient='right'))
).properties(
    title='Episode Ratings Heatmap',
    width=900,
    height=500
)

text_heatmap = alt.Chart(df).mark_text(
    baseline='middle',
    fontSize=8
).encode(
    x=alt.X('Season:O'),
    y=alt.Y('Number_season:O'),
    text=alt.Text('Rating:Q', format='.1f'),
    color=alt.condition((alt.datum.Rating >= 7.5), alt.value('black'), alt.value('white'))
).transform_filter(
    (alt.datum.Rating >= 8.5) | (alt.datum.Rating <= 6.5)
)

final_chart_1 = alt.vconcat(
    rating_chart,
    heatmap_rating + text_heatmap
).resolve_scale(
    color='independent'
)
final_chart_1

alt.VConcatChart(...)

### How have the viewers evolved over time?

Charts:

- Line chart (Superposition of average and individuals)

**Line chart:**

- Provides an overview of the overall trend, but also an overview of the individual values for each episode.

- Axes: Season (x-axis, ordinal) and Viewers (y-axis, quantitative).

- Red points and values encode the average viewers per season.

- A blue line connects the average values to help readability and to highlight the trend over time.

- Blue color points with less opacity encode the individual values. 

- Individual values at the same x-value allow us to see the variance of viewers on a season

Considered charts:

- Heatmap, as used in the previous question. Problem: the high variance in viewer values makes a continuous distribution harder to read in a grid format


In [4]:
season_avg_viewers = df.groupby("Season")["Viewers"].mean().reset_index()

scatter_viewers = alt.Chart(df).mark_circle(opacity=0.3).encode(
    x=alt.X("Season:O", title="Season", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("Viewers:Q", title="Viewers (millions)"))

line_viewers = alt.Chart(season_avg_viewers).mark_line().encode(
    x=alt.X("Season:O"),
    y=alt.Y("Viewers:Q"))

point_viewers = alt.Chart(season_avg_viewers).mark_circle(color='red').encode(
    x=alt.X("Season:O"),
    y=alt.Y("Viewers:Q"))

text_year = alt.Chart(season_avg_viewers).mark_text(dy=-10,dx=10,fontSize=10).encode(
    x=alt.X("Season:O"),
    y=alt.Y("Viewers:Q"),
    text=alt.Text('Viewers:Q', format='.1f'))

final_chart_2 = (scatter_viewers + line_viewers + point_viewers + text_year).properties(
    title="Evolution of Viewers per Season (Distribution vs. Average)",
    width=1000,
    height=600
).configure_view(
    stroke=None
)

final_chart_2

alt.LayerChart(...)

### Is there a correlation between the gradings and the viewers?  

Chart:

- Scatter plot (Single view)

**Scatter plot:**

- Allows us to see the relationship between both variables across all episodes

- Axes: Rating (x-axis, quantitative) and Viewers (y-axis, quantitative)

- Color encodes the season, providing temporal context to each point.

- Size encodes the number of votes, adding a third dimension to assess episode popularity.

- Opacity to avoid clutter. 

- A regression line allows to see the linear relationship

Considered charts:

- Heatmap. Problem: A heatmap does not show correlation directly.

In [5]:
scatter = alt.Chart(df).mark_circle(opacity=0.3,stroke='white',strokeWidth=0.5).encode(
    y=alt.Y("Viewers:Q", title="Viewers (millions)", scale=alt.Scale(domain=[0, 35])),
    x=alt.X('Rating:Q', title='Ratings',scale=alt.Scale(zero=False), axis=alt.Axis(labelAngle=0)),
    color=alt.Color('Season:Q', title='Season'),
    size=alt.Size('Votes:Q',title='Number of Votes',scale=alt.Scale(range=[10, 500])),
).properties(
    title='Relationship between Episode Ratings and Viewers',
    width=800,
    height=500,

)

regression = alt.Chart(df).transform_regression(
    'Rating', 'Viewers'
).mark_line(
    color='black',
    size=3,
    clip=True
).encode(
    x=alt.X('Rating:Q'),
    y=alt.Y('Viewers:Q', scale=alt.Scale(domain=[0, 35])),
)

final_chart_3 = (scatter + regression
).resolve_scale(
    color='independent'
).configure_view(
    stroke=None
)

final_chart_3

alt.LayerChart(...)

### Are the number of viewers for the episodes related to the weekday they were aired?

Charts:

- Bar chart and box plots (Juxtaposition)

- Vertical alignment. Same x-axis (weeakday)

**Bar chart:**

- Shows the number of episodes aired on each weekday. It provides context for how the aired episodes are distributed.

- Axes: Weekday (x-axis, ordinal) and Number of episodes (count) (y-axis, quantitative)



**Box plots:**

- Show the distribution and variance of viewers for each weekday, allowing comparison of mean and variance.

- Axis: Weekday (x-axis, ordinal) and Viewers (y-axis, quantitative)

**Considered charts:**

- Bar chart of mean viewers per weekday. Problem: it is not interpretable and can cause confusion.

In [ ]:
weekday_order = ["Monday","Tuesday","Wednesday","Thursday","Friday","Saturday","Sunday"]

episode_count = alt.Chart(df).mark_bar().encode(
    x=alt.X('Weekday:O', sort=weekday_order, axis=alt.Axis(labelAngle=0)),
    y=alt.Y('count():Q', title='Number of Episodes')
).properties(
    title='Number of Episodes Aired per Weekday',
    width=600,
    height=350
)

box = alt.Chart(df).mark_boxplot().encode(
    x=alt.X('Weekday:O', sort=weekday_order,axis=alt.Axis(labelAngle=0)),
    y=alt.Y('Viewers:Q', title='Viewers (millions)')
).properties(
    title='Distribution of Viewers by Weekday',
    width=600,
    height=450
)

final_chart_4 = alt.vconcat(
    episode_count,
    box
).configure_view(
    stroke=None
)

final_chart_4

alt.VConcatChart(...)

### Do the seasons’ number of viewers present any relevant pattern?

Charts:

- Line chart and box plots (Juxtaposition)

- Vertical alignment. Same x-axis (Season)

**Line chart:**

- Provides an overview of the overall trend, but also an overview of the individual values for each episode.

- Axes: Seasons (x-axis, ordinal) and Viewers (y-axis, quantitative).

- Red points and values encode the average viewers per season.


**Box plots:**

- Show the distribution of individual episode viewers within each season, allowing comparison of the variance across seasons and identification of outliers.

- Axes: Seasons (x-axis, ordinal) and Viewers (y-axis, quantitative).


In [7]:
season_avg_viewers = df.groupby("Season")["Viewers"].mean().reset_index()

line_chart = alt.Chart(season_avg_viewers).mark_line().encode(
    x=alt.X("Season:O",title="Season", sort="ascending", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("Viewers:Q",title="Average Viewers (millions)")
).properties(
    title="Average Number of Viewers per Season",
    width=800,
    height=250
)

text_year = alt.Chart(season_avg_viewers).mark_text(dy=-10,fontSize=10).encode(
    x=alt.X("Season:O"),
    y=alt.Y("Viewers:Q"),
    text=alt.Text('Viewers:Q', format='.1f'))

point_viewers = alt.Chart(season_avg_viewers).mark_circle(color='red', size=50).encode(
    x=alt.X("Season:O"),
    y=alt.Y("Viewers:Q")
)

boxplot = alt.Chart(df).mark_boxplot().encode(
    x=alt.X("Season:O",title="Season", sort="ascending",axis=alt.Axis(labelAngle=0)),
    y=alt.Y("Viewers:Q",title="Viewers (millions)"),
).properties(
    title="Distribution of Episode Viewers per Season",
    width=800,
    height=450
)

final_chart_5 = alt.vconcat(
    line_chart + text_year + point_viewers,
    boxplot
).configure_view(
    stroke=None
)

final_chart_5

alt.VConcatChart(...)

### Extra: Do outlier episodes present any distinct patterns?

Charts:

- Bar chart and scatter plot (Juxtaposition)

**Bar chart:**

- Shows the number of outlier episodes per season, split by type (above or below the IQR bounds).

- Axes: Season (x-axis, ordinal) and Number of Outlier Episodes (y-axis, quantitative).

- Color encodes outlier type: high (above upper bound) vs. low (below lower bound).

**Scatter plot:**

- Displays outlier episodes, reusing the same encoding as question 3, allowing comparison of whether outlier episodes cluster in specific rating or viewer ranges.

- Axes: Rating (x-axis, quantitative) and Viewers (y-axis, quantitative).

- Color encodes the outlier type.

- Size encodes the number of votes.

Assumptions:

- Outliers are defined using the IQR method.

In [8]:
def get_outliers(df, cols):
    results = []
    for col in cols:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR

        outliers = df[(df[col] < lower) | (df[col] > upper)].copy()
        outliers["Outlier_direction"] = "High"
        outliers.loc[outliers[col] < lower, "Outlier_direction"] = "Low"
        outliers["Outlier_column"] = col
        results.append(outliers)

    return pd.concat(results).reset_index(drop=True)

cols = ["Rating", "Viewers"]

outliers = get_outliers(df, cols)

bar_outliers = alt.Chart(outliers).mark_bar().encode(
    x=alt.X("Season:O", title="Season", sort="ascending", axis=alt.Axis(labelAngle=0)),
    y=alt.Y("count():Q", title="Number of Outlier Episodes"),
    color=alt.Color("Outlier_direction:N",title="Outlier Type",scale=alt.Scale(domain=["High", "Low"], range=["#1f77b4", "#d62728"]), legend=None)
).properties(
    title="Outlier Episodes per Season by Direction",
    width=800,
    height=300
)

scatter_outliers = alt.Chart(outliers).mark_circle(stroke="white", strokeWidth=0.5).encode(
    x=alt.X("Rating:Q", title="Rating", scale=alt.Scale(zero=False), axis=alt.Axis(labelAngle=0)),
    y=alt.Y("Viewers:Q", title="Viewers (millions)"),
    color=alt.Color("Outlier_direction:N",title="Outlier Type",scale=alt.Scale(domain=["High", "Low"], range=["#1f77b4", "#d62728"]), legend=alt.Legend(orient='right')),
    size=alt.Size("Votes:Q", title="Number of Votes", scale=alt.Scale(range=[10, 500]),legend=alt.Legend(orient='right'))
).properties(
    title="Outlier Episodes: Rating vs. Viewers",
    width=800,
    height=450
)

final_chart_6 = alt.vconcat(
    bar_outliers,
    scatter_outliers
).resolve_scale(
    color="independent",
    size="independent"
).configure_view(
    stroke=None
)

final_chart_6

alt.VConcatChart(...)

### Final visualization considerations

- The visualization uses seven charts, which is considered the optimal maximum.

- The first two charts (average ratings and viewers) provide an overall overview. Both use red points and labels to highlight averages. The viewers chart also shows the distribution of individual episodes.

- The scatter plot allows comparison between ratings and viewers and also highlights the outliers with color encoding. Color does not encode seasons to avoid redundancy because they are visible in the previous charts.

- The heatmap shows individual episode ratings, emphasizing extreme values with text labels for clarity.

- The next two stacked charts focus on aired weekdays. Both are aligned, share the same x-axis and are encoded with the same color, that is distinct from ratings and outliers, allowing quick comparison.

- The final chart counts the number of outliers per season. It uses the same color as the outliers in the scatter plot to create visual consistency.

- The box plot chart of viewer distribution per seasons is not used, because the distribution is captured by the average viewers chart, and outliers can be identified through the scatter and outlier count charts

In [9]:
season_avg_rating  = df.groupby("Season")["Rating"].mean().reset_index()
season_avg_viewers = df.groupby("Season")["Viewers"].mean().reset_index()
WEEKDAY_ORDER = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]

outliers_df = get_outliers(df, ["Viewers", "Rating"])

x_season  = alt.X("Season:O",  title="Season",          sort="ascending",   axis=alt.Axis(labelAngle=0))
x_weekday = alt.X("Weekday:O", title="Day of the Week",  sort=WEEKDAY_ORDER, axis=alt.Axis(labelAngle=0))

def chart_average_rating(w, h):
    base = alt.Chart(season_avg_rating).encode(
        x=x_season,
        y=alt.Y("Rating:Q", title="Average Rating", scale=alt.Scale(domain=[6, 9]))
    )
    return (
        base.mark_line(color="steelblue") +
        base.mark_circle(color="red", size=40) +
        base.mark_text(dy=-10, dx=1, fontSize=9).encode(text=alt.Text("Rating:Q", format=".1f"))
    ).properties(title="Average Rating per Season", width=w, height=h)

def chart_heatmap(w, h):
    base = alt.Chart(df).encode(
        x=alt.X("Season:O", title="Season", axis=alt.Axis(orient="top", labelAngle=0)),
        y=alt.Y("Number_season:O", title="Episode Number")
    )
    hmap = base.mark_rect(stroke="white", strokeWidth=0.5).encode(
        color=alt.Color("Rating:Q", title="Rating",
                        scale=alt.Scale(scheme="viridis"),
                        legend=alt.Legend(orient="right"))
    )
    txt = base.mark_text(baseline="middle", fontSize=7).encode(
        text=alt.Text("Rating:Q", format=".1f"),
        color=alt.condition(alt.datum.Rating >= 7.5, alt.value("black"), alt.value("white"))
    ).transform_filter((alt.datum.Rating >= 8.5) | (alt.datum.Rating <= 6.5))
    return (hmap + txt).properties(title="Individual Episode's Ratings", width=w, height=h)

def chart_average_season(w, h):
    base_avg = alt.Chart(season_avg_viewers).encode(
        x=x_season, y=alt.Y("Viewers:Q", title="Viewers (millions)")
    )
    base_pts = alt.Chart(df).encode(
        x=x_season, y=alt.Y("Viewers:Q", title="Viewers (millions)")
    )
    return (
        base_pts.mark_circle(color="steelblue", opacity=0.3) +
        base_avg.mark_line(color="steelblue") +
        base_avg.mark_circle(color="red", size=40) +
        base_avg.mark_text(dy=-8, dx=5, fontSize=9).encode(text=alt.Text("Viewers:Q", format=".1f"))
    ).properties(title="Viewership per Season (Distribution and Average)", width=w, height=h)

def chart_scatter(w, h):
    df_flagged = df.copy()
    df_flagged["Type"] = df_flagged["ID"].isin(outliers_df["ID"]).map(
        {True: "Outlier", False: "No outlier"}
    )
    base = alt.Chart(df_flagged).encode(
        x=alt.X("Rating:Q", title="Rating", scale=alt.Scale(zero=False)),
        y=alt.Y("Viewers:Q", title="Viewers (millions)", scale=alt.Scale(domain=[0,35])),
        color=alt.Color("Type:N", scale=alt.Scale(
            domain=["No outlier", "Outlier"],
            range=["steelblue", "darkorange"]
        )),
        size=alt.Size("Votes:Q", title="Votes", scale=alt.Scale(range=[10, 400]))
    )
    scatter = base.mark_circle(stroke="white", strokeWidth=0.5, opacity=0.5)
    regression = alt.Chart(df).transform_regression("Rating", "Viewers").mark_line(
        color="black", size=2, clip=True
    ).encode(x=alt.X("Rating:Q"), y=alt.Y("Viewers:Q",scale=alt.Scale(zero=True)))
    return (scatter + regression).properties(title="Ratings vs Viewers with Highlighted Outliers", width=w, height=h)

def chart_weekday(w, h):
    bar = alt.Chart(df).mark_bar(color="#9ECAE1").encode(
        x=x_weekday,
        y=alt.Y("count():Q", title="Number of Episodes", )
    ).properties(title="Number of Episodes Aired per Weekday", width=w, height=h)
    box = alt.Chart(df).mark_boxplot(color="#9ECAE1").encode(
        x=x_weekday,
        y=alt.Y("Viewers:Q", title="Viewers (millions)")
    ).properties(title="Distribution of Viewers by Weekday", width=w, height=h)
    return alt.vconcat(bar, box).resolve_scale(x="shared")

def chart_outlier(w, h):
    return alt.Chart(outliers_df).mark_bar(color="darkorange", opacity=0.5).encode(
        x=x_season,
        y=alt.Y("count():Q", title="Outlier Episodes")
    ).properties(title="Outlier Episodes Count per Season", width=w, height=h)

final = (
    alt.vconcat(
        alt.hconcat(
            chart_average_rating(400, 250),
            chart_average_season(500, 250),
            chart_scatter(400, 250)
        ).resolve_scale(color="independent", size="independent"),
        alt.hconcat(
            chart_heatmap(700, 500),
            chart_weekday(300, 200),
            chart_outlier(250, 200)
        ).resolve_scale(color="independent"),
    )
    .resolve_scale(color="independent")
    .configure_view(stroke=None)
    .configure_axis(labelFontSize=9, titleFontSize=10)
    .configure_title(fontSize=11)
)

final

alt.VConcatChart(...)